# Phase 1 — Generate answers & score claims (Kaggle GPU)

Runs **once** on a GPU. For each K-QA question: generate an answer → split into atomic claims → score each with P(true) → cache to `claims_phase1.jsonl`.
Phases 2–3 then run on that cache with no GPU.

### Before running
1. Settings → **GPU** on and **Internet** on.
2. Add your **code dataset** (the one containing the `sac/` folder) via **+ Add Input**.
   Run `!ls /kaggle/input` and set `SAC_PATH` below to the folder that contains `sac/`.
3. Gated Llama-3 only: add an `HF_TOKEN` secret (Add-ons → Secrets) and uncomment the login lines in step 2.

In [ ]:
!pip -q install -U transformers bitsandbytes accelerate

# ---------- Config ----------
SAC_PATH    = "/kaggle/input/severity-aware-conformal"   # folder that contains the sac/ package
N_QUESTIONS = None    # None = all 201 K-QA questions; set an int to subsample
CACHE       = "/kaggle/working/claims_phase1.jsonl"
# ----------------------------

import sys
sys.path.insert(0, SAC_PATH)

from sac.kqa_loader import load_kqa
from sac.hf_backend import HFBackend
from sac.decompose import decompose
from sac.scoring import score_claim
from sac.cache import append_claims, existing_claim_ids, load_claims
from sac.crc import Claim

## 1. Load K-QA questions (with gold answers)

In [ ]:
# K-QA gold-answer file: Question + Must_have / Nice_to_have statements
!wget -q -O /kaggle/working/kqa.jsonl https://raw.githubusercontent.com/Itaymanes/K-QA/main/dataset/questions_w_answers.jsonl

items = load_kqa("/kaggle/working/kqa.jsonl")[:N_QUESTIONS]
print(f"{len(items)} questions | gold statements in first item: {len(items[0].statements)}")

## 2. Load the model

In [ ]:
# Gated Llama-3 only: uncomment after adding an HF_TOKEN secret
# from huggingface_hub import login
# from kaggle_secrets import UserSecretsClient
# login(token=UserSecretsClient().get_secret("HF_TOKEN"))

backend = HFBackend()

## 3. Generate → decompose → score  (checkpointed, resumable)

In [ ]:
done = existing_claim_ids(CACHE)
for it in items:
    if any(c.startswith(it.qid + "_") for c in done):
        continue                                  # already cached -> skip
    answer = backend.generate(f"Question: {it.question}\nAnswer:")
    claims = [
        Claim(text=ct, confidence=score_claim(ct, backend),
              answer_id=it.qid, claim_id=f"{it.qid}_c{j}")
        for j, ct in enumerate(decompose(it.question, answer, backend))
    ]
    append_claims(CACHE, claims)                  # checkpoint after each question
    print(f"{it.qid}: {len(claims)} claims")

## 4. Inspect the cached claims

In [ ]:
claims = load_claims(CACHE)
print(f"total: {len(claims)} claims from {len(set(c.answer_id for c in claims))} answers\n")
for c in claims[:10]:
    print(f"  conf={c.confidence:.2f}  {c.text[:75]}")

# Persist for Phase 2: 'Save Version', or copy /kaggle/working/claims_phase1.jsonl to a Kaggle Dataset.